# Chunking Techniques: Breaking Documents into Searchable Pieces

## What You'll Learn

We're about to collect 5 chunking strategies. Each one goes in your toolkit.

- Why splitting documents into smaller pieces is not optional — it's fundamental
- Fixed-size chunking (the baseline every system starts with)
- Overlapping chunks (the simple fix that prevents most failures)
- Sentence-based chunking (respecting the boundaries language already gives us)
- Semantic chunking (splitting by topic, not by character count)
- Recursive chunking (LangChain's workhorse approach)
- How to choose the right strategy for your use case

**Prerequisites:** Understanding of RAG ([Notebook 01](01_what_is_rag.ipynb))

In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="rag",
    notebook_name="02_chunking_techniques.ipynb"
)

---

## Why Chunking Is the Step Everyone Gets Wrong

Here's a surprising fact: in production RAG systems, the single biggest lever for improving answer quality is not the LLM, not the embedding model, and not the vector database. It's the chunking strategy.

Most teams treat chunking as a footnote. The teams that ship great RAG products treat it as a first-class engineering problem.

Let's understand why it matters so much.

### Analogy: Your Brain and a Library Card Catalog

Imagine a library where every book is locked in a single enormous filing cabinet. When you ask a question, the librarian has to haul out the entire cabinet and sort through it. Not helpful.

Now imagine the same library, except every chapter has its own index card with a one-sentence summary. When you ask "What did Einstein say about time dilation?" the librarian pulls out exactly the right card and hands you the right chapter.

What the index card system gets right: small units with precise summaries let you retrieve the right piece without dragging along everything else.

Where the analogy breaks down: in a real library, a human wrote those index cards with understanding. In RAG, we generate embeddings automatically — and the quality of that embedding depends heavily on the size and coherence of the chunk.

### Why chunking matters in plain terms

Think of an embedding as a one-sentence summary of a chunk. If the chunk is a single focused paragraph about Venus, the summary will be precise: "facts about Venus's atmosphere." If the chunk is a 50-page PDF about the solar system, the summary becomes mush: "something vague about space."

Here's what follows from that:

- **LLMs have limited context windows.** You can only hand so much text to the LLM at query time. You want every token to earn its place.
- **Smaller chunks produce more precise embeddings.** A precise embedding means the retrieval step finds the right piece instead of a noisy mix.
- **If a chunk is too big, the embedding becomes a blurry average of many topics.** The vector drifts toward the center of too many ideas at once and is precise about none of them.
- **Each chunk gets its own embedding — its own GPS coordinate in meaning-space.** You want that coordinate to point to one clear location, not a smear across the map.

Chunking is one of the biggest levers you can pull to improve RAG quality — production systems at companies like Google and OpenAI spend significant engineering time on this before touching anything else.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# === Left side: One huge document → one blurry embedding ===
ax1.set_title('❌ Without Chunking', fontsize=14, fontweight='bold', color='#c0392b')
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 10)
ax1.axis('off')

# Big document blob
big_doc = patches.FancyBboxPatch((0.5, 4), 4, 5, boxstyle='round,pad=0.3',
                                  facecolor='#e74c3c', alpha=0.3, edgecolor='#c0392b', linewidth=2)
ax1.add_patch(big_doc)
ax1.text(2.5, 6.5, '📄 Entire\nDocument\n(100 pages)', ha='center', va='center', fontsize=11, fontweight='bold')

# Arrow
ax1.annotate('', xy=(7, 6.5), xytext=(5, 6.5),
             arrowprops=dict(arrowstyle='->', lw=2, color='gray'))

# Blurry embedding
blurry = patches.Circle((8, 6.5), 1.2, facecolor='#e74c3c', alpha=0.15, edgecolor='#c0392b',
                         linewidth=2, linestyle='--')
ax1.add_patch(blurry)
ax1.text(8, 6.5, '🔴\nBlurry\nEmbedding', ha='center', va='center', fontsize=9, color='#c0392b')

ax1.text(5, 2.5, '"What is this document about?"\n→ A little bit of everything...\n→ Not useful for finding specifics!',
         ha='center', va='center', fontsize=10, style='italic',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#fadbd8', edgecolor='#c0392b'))

# === Right side: Split into chunks → precise embeddings ===
ax2.set_title('✅ With Chunking', fontsize=14, fontweight='bold', color='#27ae60')
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.axis('off')

# Multiple small chunks
colors = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']
labels = ['Chunk 1:\nHistory', 'Chunk 2:\nScience', 'Chunk 3:\nMath', 'Chunk 4:\nArt']
for i, (color, label) in enumerate(zip(colors, labels)):
    y = 8.5 - i * 1.8
    chunk = patches.FancyBboxPatch((0.3, y - 0.5), 2.5, 1.2, boxstyle='round,pad=0.2',
                                   facecolor=color, alpha=0.3, edgecolor=color, linewidth=2)
    ax2.add_patch(chunk)
    ax2.text(1.55, y + 0.1, label, ha='center', va='center', fontsize=9, fontweight='bold')

    # Arrow
    ax2.annotate('', xy=(5.5, y + 0.1), xytext=(3, y + 0.1),
                 arrowprops=dict(arrowstyle='->', lw=1.5, color='gray'))

    # Precise embedding dot
    emb = patches.Circle((6.5 + (i % 2) * 1.5, y + 0.1), 0.4, facecolor=color, alpha=0.5,
                          edgecolor=color, linewidth=2)
    ax2.add_patch(emb)
    ax2.text(6.5 + (i % 2) * 1.5, y + 0.1, '🎯', ha='center', va='center', fontsize=10)

ax2.text(7.2, 1.5, 'Each chunk has a\nPRECISE embedding!\n→ Easy to find exactly\n   what you need!',
         ha='center', va='center', fontsize=10, style='italic',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='#d5f5e3', edgecolor='#27ae60'))

plt.suptitle('Why Chunking Matters for RAG', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---

## Method 1: Fixed-Size Chunking

Have you ever photocopied a book by just pressing "Copy All" and watching the machine spit out pages without caring if a sentence got cut between pages? That's fixed-size chunking.

The approach: split every N characters, no matter what.

Analogy: cutting a pizza into equal slices regardless of where the toppings are — you might cut right through a pepperoni. What the analogy gets right: the slices are predictable and you always know how many you're getting. Where it breaks down: a pizza slice with half a pepperoni is still edible. A chunk that contains half a sentence produces a damaged embedding that degrades retrieval quality.

Fair warning: the code below catches chunks that start or end mid-sentence — watch the warnings to see how often this happens even on a short paragraph.

| Pros | Cons |
|------|------|
| Simple to implement | Can cut sentences in half |
| Predictable chunk sizes | May split related information |
| Fast — no language processing needed | Chunks may start or end mid-thought |

Fixed-size chunking is the starting point for almost every RAG system. The next four methods are progressively smarter fixes for its limitations.

In [ ]:
def fixed_size_chunking(text, chunk_size=200):
    """Split text into fixed-size chunks of `chunk_size` characters."""
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i + chunk_size])
    return chunks


# Sample text about space
sample_text = (
    "The solar system is a fascinating place that has captivated human imagination for thousands of years. "
    "Our Sun, a medium-sized star, sits at the center and provides light and warmth to eight planets. "
    "Mercury is the closest planet to the Sun and has extreme temperature swings. "
    "Venus is often called Earth's twin because of its similar size, but its atmosphere is incredibly hostile. "
    "Earth is the only known planet to support life, with its perfect distance from the Sun and protective atmosphere. "
    "Mars, the Red Planet, has been the target of numerous space missions searching for signs of past life. "
    "Jupiter is the largest planet, a gas giant with a famous Great Red Spot storm that has raged for centuries. "
    "Saturn is known for its beautiful ring system made of ice and rock particles. "
    "Uranus and Neptune are the ice giants of the outer solar system, mysterious and far from the Sun."
)

print(f"📄 Original text length: {len(sample_text)} characters")
print("=" * 60)

chunks = fixed_size_chunking(sample_text, chunk_size=200)

for i, chunk in enumerate(chunks):
    # Check if chunk starts or ends mid-sentence
    starts_mid = (i > 0 and not chunk[0].isupper() and chunk[0] != ' ')
    ends_mid = (i < len(chunks) - 1 and not chunk.rstrip().endswith(('.', '!', '?')))

    warning = ""
    if starts_mid:
        warning += " ⚠️ Starts mid-sentence!"
    if ends_mid:
        warning += " ⚠️ Ends mid-sentence!"

    print(f"\n📦 Chunk {i + 1} ({len(chunk)} chars){warning}")
    print(f"   \"{chunk}\"")

print("\n" + "=" * 60)
print(f"Total chunks: {len(chunks)}")
print("\n⚠️ Notice how some chunks cut right through sentences!")
print("   This means a search about Venus might miss half the information.")

---

## Method 2: Overlapping Chunks

Here is the problem that fixed-size chunking creates, stated concretely: imagine a sentence that reads "The patient developed severe side effects including nausea, and the treatment was immediately discontinued." If that sentence gets split between chunk 4 and chunk 5, a doctor searching for "treatment discontinued" only gets the second half, without the cause. That's a retrieval failure.

Overlapping chunks fix this by making each chunk share some text with its neighbors.

Analogy: shingles on a roof. Roofing tiles overlap so that rain can't fall through the gap between them. Overlapping chunks ensure no piece of information falls through the gap between chunks. What the analogy gets right: coverage is complete — every piece of text appears in at least one full chunk. Where it breaks down: shingles overlap because physics forces them to. With chunks, you choose the overlap amount, and too much overlap bloats your index and slows retrieval.

Typical overlap: 10-20% of chunk size. For chunk_size=200, use overlap=20 to 40 characters.

In [ ]:
def overlapping_chunking(text, chunk_size=200, overlap=50):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap  # Step back by `overlap` characters
    return chunks


print("🔗 Overlapping Chunking (chunk_size=200, overlap=50)")
print("=" * 60)

overlap_chunks = overlapping_chunking(sample_text, chunk_size=200, overlap=50)

for i, chunk in enumerate(overlap_chunks):
    print(f"\n📦 Chunk {i + 1} ({len(chunk)} chars)")
    print(f"   \"{chunk}\"")

    # Show overlap with next chunk
    if i < len(overlap_chunks) - 1:
        overlap_text = chunk[-50:] if len(chunk) >= 50 else chunk
        print(f"   🔗 Overlap region (last 50 chars): \"{overlap_text}\"")

print("\n" + "=" * 60)
print(f"Total chunks: {len(overlap_chunks)} (vs {len(chunks)} without overlap)")
print("\n✅ Now boundary information is preserved in overlapping regions!")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 4))

text_len = len(sample_text)

# === Top: Fixed chunks (with gaps highlighted) ===
ax1.set_title('❌ Fixed-Size Chunks — Information Lost at Boundaries', fontsize=12, fontweight='bold')
ax1.set_xlim(0, text_len)
ax1.set_ylim(0, 2)
ax1.axis('off')

colors_fixed = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6']
chunk_size = 200
for i, start in enumerate(range(0, text_len, chunk_size)):
    end = min(start + chunk_size, text_len)
    color = colors_fixed[i % len(colors_fixed)]
    rect = patches.FancyBboxPatch((start + 2, 0.4), end - start - 4, 1.2,
                                  boxstyle='round,pad=0.05',
                                  facecolor=color, alpha=0.4, edgecolor=color, linewidth=2)
    ax1.add_patch(rect)
    ax1.text((start + end) / 2, 1.0, f'Chunk {i + 1}', ha='center', va='center',
             fontsize=9, fontweight='bold')

    # Mark boundary gaps
    if i > 0:
        ax1.axvline(x=start, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
        ax1.text(start, 0.15, '⚡', ha='center', fontsize=10)

# === Bottom: Overlapping chunks ===
ax2.set_title('✅ Overlapping Chunks — Boundaries Are Covered', fontsize=12, fontweight='bold')
ax2.set_xlim(0, text_len)
ax2.set_ylim(0, 2)
ax2.axis('off')

overlap = 50
start = 0
i = 0
while start < text_len:
    end = min(start + chunk_size, text_len)
    color = colors_fixed[i % len(colors_fixed)]

    # Main chunk body
    rect = patches.FancyBboxPatch((start + 2, 0.4), end - start - 4, 1.2,
                                  boxstyle='round,pad=0.05',
                                  facecolor=color, alpha=0.3, edgecolor=color, linewidth=2)
    ax2.add_patch(rect)
    ax2.text((start + end) / 2, 1.0, f'Chunk {i + 1}', ha='center', va='center',
             fontsize=9, fontweight='bold')

    # Highlight overlap region
    if i > 0:
        overlap_start = start
        overlap_end = start + overlap
        overlap_rect = patches.FancyBboxPatch((overlap_start, 0.3), overlap, 1.4,
                                              boxstyle='round,pad=0.02',
                                              facecolor='#f1c40f', alpha=0.4,
                                              edgecolor='#f39c12', linewidth=1.5,
                                              linestyle='--')
        ax2.add_patch(overlap_rect)
        ax2.text((overlap_start + overlap_end) / 2, 0.12, '🔗', ha='center', fontsize=8)

    start = end - overlap
    i += 1

plt.tight_layout()
plt.show()

---

## Method 3: Sentence-Based Chunking

What if instead of fighting against the sentence boundaries in natural language, we worked with them?

Every sentence is a complete thought. Why not respect that?

Sentence-based chunking groups complete sentences together until the total length reaches a target size. You never cut mid-sentence. The price you pay: chunk sizes become variable, because some sentences are short and some are long.

Analogy: sorting Lego pieces into bags. You don't cut pieces in half to make the bags equal — you fill each bag with whole pieces until it's full, then start a new bag. What the analogy gets right: every piece stays intact, and the bag is always usable. Where it breaks down: Lego bags are independent. In text, the last sentence of one chunk often sets up the first sentence of the next, so you lose that connection.

This is why sentence-based chunking works well for text that is already self-contained — encyclopedias, product descriptions, FAQ pages — and works less well for narrative text where one sentence flows into the next.

In [ ]:
def sentence_chunking(text, max_chunk_size=300):
    """Split text into chunks that respect sentence boundaries."""
    # Simple sentence splitting (handles . ! ?)
    sentences = text.replace('! ', '!|').replace('? ', '?|').replace('. ', '.|').split('|')
    chunks = []
    current = ''
    for s in sentences:
        if len(current) + len(s) > max_chunk_size and current:
            chunks.append(current.strip())
            current = s
        else:
            current += ' ' + s
    if current.strip():
        chunks.append(current.strip())
    return chunks


print("📝 Sentence-Based Chunking (max_chunk_size=300)")
print("=" * 60)

sent_chunks = sentence_chunking(sample_text, max_chunk_size=300)

for i, chunk in enumerate(sent_chunks):
    ends_clean = chunk.rstrip().endswith(('.', '!', '?'))
    status = "✅ Complete sentences" if ends_clean else "⚠️ Partial sentence"
    print(f"\n📦 Chunk {i + 1} ({len(chunk)} chars) — {status}")
    print(f"   \"{chunk}\"")

print("\n" + "=" * 60)
print("\n🔍 Comparison with Fixed-Size Chunking:")
print(f"   Fixed-size chunks: {len(chunks)} chunks (many cut mid-sentence)")
print(f"   Sentence chunks:   {len(sent_chunks)} chunks (all complete sentences!)")
print("\n✅ Every chunk contains complete, coherent sentences!")

---

## Method 4: Semantic Chunking

What if we stopped splitting text based on size entirely, and instead split based on when the topic changes?

That's the idea behind semantic chunking.

Analogy: imagine you're reading a Wikipedia article that starts talking about Marie Curie's childhood, then pivots to her scientific discoveries, then covers her personal life. A smart editor would put those three sections in separate documents if they were going to file them. Semantic chunking is that smart editor — it detects the pivot points automatically. What the analogy gets right: the boundaries are placed where meaning actually shifts, not at some arbitrary character count. Where it breaks down: a human editor knows from experience when a topic has truly changed. The algorithm uses similarity scores and a threshold, so it can be fooled by gradual topic drift or miss sharp pivots when adjacent sentences happen to share vocabulary.

How it works, step by step:

1. Split the text into individual sentences.
2. Compute an embedding (a meaning vector) for each sentence.
3. For each adjacent pair of sentences, compute cosine similarity.
4. When the similarity drops below a threshold, that drop signals a topic change.
5. Cut the chunk at that point.

The threshold is a hyperparameter. Set it too high and you get many small chunks. Set it too low and you merge unrelated topics.

Fair warning: the implementation below uses Jaccard similarity (word overlap) instead of true embedding similarity, because we want this notebook to run without downloading any ML models. Real semantic chunking uses embedding-based cosine similarity and produces meaningfully better boundaries.

Semantic chunking is used in production RAG systems where retrieval precision matters most — legal document search, medical records, and customer support knowledge bases are places where teams invest in this level of chunking sophistication.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulated similarity scores between adjacent sentence pairs
# (In real life, these come from comparing embedding vectors)
sentence_labels = [
    'S1-S2\n(both about\nsolar system)', 
    'S2-S3\n(Sun to\nMercury)',
    'S3-S4\n(Mercury to\nVenus)', 
    'S4-S5\n(Venus to\nEarth)',
    'S5-S6\n(Earth to\nMars)',
    'S6-S7\n(Mars to\nJupiter)', 
    'S7-S8\n(Jupiter to\nSaturn)',
    'S8-S9\n(Saturn to\nice giants)'
]

# Fake but illustrative: high similarity within topics, drops at boundaries
similarities = [0.85, 0.35, 0.72, 0.68, 0.65, 0.25, 0.78, 0.30]
threshold = 0.4

fig, ax = plt.subplots(figsize=(14, 5))

x = np.arange(len(similarities))
colors = ['#2ecc71' if s >= threshold else '#e74c3c' for s in similarities]

bars = ax.bar(x, similarities, color=colors, alpha=0.7, edgecolor='black', linewidth=1)
ax.axhline(y=threshold, color='red', linestyle='--', linewidth=2, label=f'Threshold = {threshold}')

# Annotate chunk boundaries
for i, sim in enumerate(similarities):
    if sim < threshold:
        ax.annotate('✂️ CHUNK\nBOUNDARY', xy=(i, sim), xytext=(i, sim + 0.15),
                    fontsize=10, fontweight='bold', color='#c0392b', ha='center',
                    arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2))
    else:
        ax.text(i, sim + 0.03, f'{sim:.2f}', ha='center', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(sentence_labels, fontsize=8)
ax.set_ylabel('Similarity Score', fontsize=12)
ax.set_xlabel('Adjacent Sentence Pairs', fontsize=12)
ax.set_title('🧠 Semantic Chunking: Detecting Topic Boundaries\n'
             'Low similarity = topic change = chunk boundary!', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11, loc='upper right')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def jaccard_similarity(s1, s2):
    """Simple word-overlap similarity (no ML needed!)."""
    w1 = set(s1.lower().split())
    w2 = set(s2.lower().split())
    if not (w1 | w2):
        return 0
    return len(w1 & w2) / len(w1 | w2)


def semantic_chunking(text, threshold=0.1):
    """Split text into chunks based on semantic similarity between sentences."""
    sentences = text.replace('! ', '!|').replace('? ', '?|').replace('. ', '.|').split('|')
    sentences = [s.strip() for s in sentences if s.strip()]
    
    if not sentences:
        return []
    
    chunks = []
    current_chunk = [sentences[0]]
    
    for i in range(1, len(sentences)):
        sim = jaccard_similarity(sentences[i - 1], sentences[i])
        if sim < threshold:
            # Topic changed! Start a new chunk
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentences[i]]
        else:
            # Same topic, continue current chunk
            current_chunk.append(sentences[i])
    
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks


# A text with clear topic changes
multi_topic_text = (
    "Photosynthesis is the process by which plants convert sunlight into energy. "
    "Plants use chlorophyll in their leaves to absorb light from the sun. "
    "The light energy is used to convert carbon dioxide and water into glucose and oxygen. "
    "Basketball was invented by James Naismith in 1891 in Springfield, Massachusetts. "
    "The game is played with two teams of five players each on a rectangular court. "
    "Players score by shooting a ball through the opponent's hoop mounted ten feet high. "
    "Python is a popular programming language created by Guido van Rossum. "
    "It emphasizes code readability and allows programmers to express concepts in fewer lines. "
    "Python is widely used in web development, data science, and artificial intelligence."
)

print("🧠 Semantic Chunking Demo")
print("=" * 60)
print("\nInput text has 3 topics: 🌱 Photosynthesis, 🏀 Basketball, 🐍 Python")
print()

# Show sentence-by-sentence similarities
sentences = multi_topic_text.replace('! ', '!|').replace('? ', '?|').replace('. ', '.|').split('|')
sentences = [s.strip() for s in sentences if s.strip()]

print("📊 Similarity between adjacent sentences:")
for i in range(1, len(sentences)):
    sim = jaccard_similarity(sentences[i - 1], sentences[i])
    indicator = "🔗 Same topic" if sim >= 0.1 else "✂️ TOPIC CHANGE"
    print(f"   S{i}→S{i + 1}: similarity = {sim:.3f}  {indicator}")

print("\n" + "-" * 60)
print("\n📦 Resulting chunks:")
sem_chunks = semantic_chunking(multi_topic_text, threshold=0.1)

topic_emojis = ['🌱', '🏀', '🐍', '📌', '📌']
for i, chunk in enumerate(sem_chunks):
    emoji = topic_emojis[i] if i < len(topic_emojis) else '📌'
    print(f"\n{emoji} Chunk {i + 1} ({len(chunk)} chars):")
    print(f"   \"{chunk}\"")

print(f"\n✅ Semantic chunking correctly identified {len(sem_chunks)} topic groups!")

---

## Method 5: Recursive Chunking

Here's a practical observation: most real documents are already hierarchically organized. A research paper has sections. A recipe has steps. A textbook has chapters, then sections, then paragraphs. Why not use that structure?

Recursive chunking does exactly this. It tries the most meaningful separators first and only falls back to less meaningful ones when a chunk is still too big.

The separator hierarchy, from most meaningful to least:
1. `\n\n` (paragraph breaks) — a writer's signal that one idea ends and another begins
2. `\n` (line breaks) — a weaker structural signal, but still intentional
3. `. ` (sentence endings) — the next natural boundary
4. ` ` (spaces between words) — last resort, produces fragments

Analogy: imagine you're trying to sort a pile of loose papers from different documents. First you sort by document (the biggest natural grouping). Within each document, you sort by section. Within each section, by paragraph. You only break a paragraph apart if it's still too thick to fit in the folder. What the analogy gets right: you preserve the largest meaningful groupings and only subdivide when necessary. Where it breaks down: a folder has a fixed physical capacity. Your chunk size limit is a soft design choice — you get to pick it based on your embedding model's effective context size.

This is the approach LangChain's `RecursiveCharacterTextSplitter` implements. It is the most widely used chunking method in production RAG systems today, because it gracefully handles the variety of document structures that real-world corpora contain.

In [ ]:
def recursive_chunking(text, chunk_size=200, separators=None):
    """Split text recursively using a hierarchy of separators."""
    if separators is None:
        separators = ['\n\n', '\n', '. ', ' ']
    
    # Base case: text fits in one chunk
    if len(text) <= chunk_size:
        return [text.strip()] if text.strip() else []
    
    # Try each separator in order (most meaningful first)
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            chunks = []
            current = ''
            
            for part in parts:
                # Add separator back (except for spaces)
                test_addition = part if sep == ' ' else part + sep.rstrip()
                
                if len(current) + len(test_addition) + len(sep) <= chunk_size:
                    current = current + sep + part if current else part
                else:
                    if current.strip():
                        chunks.append(current.strip())
                    current = part
            
            if current.strip():
                chunks.append(current.strip())
            
            # Recursively split any chunks that are still too big
            remaining_seps = separators[separators.index(sep) + 1:]
            final_chunks = []
            for chunk in chunks:
                if len(chunk) > chunk_size and remaining_seps:
                    final_chunks.extend(recursive_chunking(chunk, chunk_size, remaining_seps))
                else:
                    final_chunks.append(chunk)
            
            return final_chunks
    
    # Fallback: hard split (shouldn't reach here often)
    return fixed_size_chunking(text, chunk_size)


# Multi-paragraph text with clear structure
structured_text = """Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn from data. Instead of being explicitly programmed, these systems identify patterns and make decisions with minimal human intervention.

Supervised Learning

In supervised learning, the algorithm learns from labeled training data. Each example in the training set has an input and a desired output. The algorithm learns a function that maps inputs to outputs. Common examples include email spam detection and image classification.

Unsupervised Learning

Unsupervised learning works with data that has no labels. The algorithm tries to find hidden patterns or structures in the data. Clustering and dimensionality reduction are popular unsupervised techniques. Customer segmentation is a real-world application.

Reinforcement Learning

Reinforcement learning involves an agent that learns by interacting with an environment. The agent receives rewards or penalties based on its actions. Over time, it learns a strategy that maximizes cumulative reward. Game playing AI and robotics use this approach."""

print("Recursive Chunking Demo")
print("=" * 60)
print(f"\nOriginal text: {len(structured_text)} characters")
print(f"   Contains {structured_text.count(chr(10) + chr(10))} paragraph breaks (\\n\\n)")

rec_chunks = recursive_chunking(structured_text, chunk_size=250)

print(f"\nResult: {len(rec_chunks)} chunks\n")

for i, chunk in enumerate(rec_chunks):
    print(f"--- Chunk {i + 1} ({len(chunk)} chars) ---")
    print(chunk)
    print()

print("Notice how recursive chunking respects paragraph structure!")
print("Each chunk tends to cover one coherent topic.")

---

## The Goldilocks Problem: How Chunk Size Affects Everything

Have you ever tried to describe a friend using just one word? It's nearly impossible — you lose almost everything important. Now imagine describing them in one sentence. Better, but still a lot of detail falls away. A paragraph? Now you can actually capture something real.

Chunk size works the same way. Each chunk has one embedding — one "meaning coordinate" in vector space. That coordinate needs to be precise enough to distinguish this chunk from 10,000 others, but complete enough to actually contain a useful answer.

Analogy: think of a chunk's embedding like a mailing address. "USA" is not a useful address — too vague to find anyone. "123 Main Street, Springfield, IL 62701" is useful. "The crack in the second brick on the left side of the third step" is too specific — your delivery driver can't work with that. What the analogy gets right: there's a resolution that's right for the job. Where it breaks down: unlike addresses, what counts as "right" depends on your query distribution, not just the document.

The numbers below are illustrative of the principle. Real optimal chunk sizes vary by domain, embedding model, and query type — which is why you should always verify against your own evaluation data.

- **Too small (under 100 tokens):** The embedding represents an incomplete thought. A chunk that says "water evaporates" gives the retrieval system nothing to work with for a query about the water cycle.
- **Too large (over 1000 tokens):** The embedding averages over too many ideas and ends up representing none of them well. Retrieval precision collapses.
- **Just right (100-500 tokens for most domains):** One coherent idea per chunk. The embedding lands in a precise, findable location in vector space.

The 200-500 token sweet spot is where most production RAG teams land after tuning. Treat it as a starting point, not a rule.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Chunk sizes and synthetic retrieval quality scores
chunk_sizes = [50, 100, 200, 500, 1000, 2000]
quality_scores = [0.35, 0.62, 0.88, 0.92, 0.65, 0.38]

fig, ax = plt.subplots(figsize=(12, 6))

# Color bars based on quality
colors = []
for score in quality_scores:
    if score >= 0.8:
        colors.append('#2ecc71')  # Green - good
    elif score >= 0.6:
        colors.append('#f39c12')  # Orange - okay
    else:
        colors.append('#e74c3c')  # Red - bad

bars = ax.bar([str(s) for s in chunk_sizes], quality_scores, color=colors, 
              alpha=0.8, edgecolor='black', linewidth=1.5, width=0.6)

# Add value labels on bars
for bar, score in zip(bars, quality_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{score:.0%}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Zone annotations
ax.annotate('❄️ Too Small!\nLacks context', xy=(0, 0.35), xytext=(0, 0.55),
            fontsize=11, ha='center', color='#c0392b', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2))

ax.annotate('🔥 Too Big!\nToo general', xy=(5, 0.38), xytext=(5, 0.55),
            fontsize=11, ha='center', color='#c0392b', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='#c0392b', lw=2))

# Highlight the sweet spot
ax.axvspan(1.5, 3.5, alpha=0.1, color='green')
ax.text(2.5, 1.0, '🎯 SWEET SPOT\n(200-500 tokens)', ha='center', va='center',
        fontsize=14, fontweight='bold', color='#27ae60',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='#27ae60', alpha=0.9))

ax.set_xlabel('Chunk Size (tokens)', fontsize=13, fontweight='bold')
ax.set_ylabel('Retrieval Quality', fontsize=13, fontweight='bold')
ax.set_title('📏 The Goldilocks Zone: Finding the Right Chunk Size', fontsize=15, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---

## Strategy Comparison

| Method | Complexity | Pros | Cons | Best For |
|--------|-----------|------|------|----------|
| **Fixed-size** | Low | Simple, fast, no dependencies | Cuts sentences mid-thought; silent quality loss | Quick prototypes, baselines |
| **Overlapping** | Low | Fixes boundary failures; easy to tune | Increases index size; some redundancy | Most production systems |
| **Sentence-based** | Medium | Clean sentence boundaries; natural units | Variable chunk size; sentence splitting can be fragile | Well-structured prose |
| **Semantic** | High | Topic-coherent chunks; best retrieval precision | Requires embedding infrastructure; threshold tuning | High-value retrieval, legal/medical |
| **Recursive** | Medium | Respects document hierarchy; handles mixed content | More complex code; depends on document having structure | Structured documents (PDFs, Markdown) |

You now have 5 chunking strategies in your toolkit. Use them in order of cost — start cheap, upgrade when your measurements tell you to.

## How to Choose Your Strategy

The choice isn't about finding the "best" strategy in the abstract. It's about matching the strategy to your constraints right now and knowing when to upgrade.

**For a new system with one day of resources:**
Start with overlapping fixed-size chunks. Chunk size 300 tokens, overlap 15%. No external dependencies. You can always upgrade.

**For structured documents (PDFs, Markdown with headers):**
Use recursive chunking. It respects the document's intended hierarchy, which means your chunks are more likely to cover exactly one coherent topic.

**For maximum retrieval precision:**
Use semantic chunking — but only after you've measured that you have a retrieval precision problem. Semantic chunking is expensive to build and operate. Don't reach for it until your evaluation data shows you need it.

**Always test with your own data.** The 200-500 token sweet spot is a starting point, not a rule. Your query distribution determines the right chunk size, and only your evaluation set can measure that.

**Consider what lives next to your chunks.** Store rich metadata with each chunk: document ID, source filename, page number, section header if detectable. This metadata enables filtering and makes future upgrades cheaper — you can re-embed with a new strategy without re-parsing your entire corpus.

---

## Key Takeaways

1. Chunking is the biggest single lever for RAG quality that most teams underinvest in. The embedding model and vector database matter less than most people expect; the chunk strategy matters more.

2. Fixed-size chunking is a known-broken baseline. It produces silent retrieval failures when sentences span chunk boundaries. Use overlapping chunks from the start.

3. Overlapping chunks solve the boundary problem at low cost. 10-20% overlap eliminates most boundary failures without significantly increasing index size.

4. Sentence-based chunking respects the natural units of language. Use it when your text is well-written and self-contained. It won't help for narrative text with heavy coreference.

5. Semantic chunking produces the most topic-coherent chunks. It's the right choice when retrieval precision is your primary constraint and you have the infrastructure to support it.

6. Recursive chunking handles the real world: documents with mixed structure, varying section lengths, and multiple levels of hierarchy.

7. Chunk size of 200-500 tokens is where most teams land. Treat it as a starting point, measure against your eval set, and tune from there.

8. Metadata matters as much as the chunk boundary algorithm. Store it from day one.

---

## 🤔 Test Your Understanding

**Question 1:** Why can't we just embed an entire document as one big chunk?

<details>
<summary>👉 Click to reveal answer</summary>

When you embed a huge document as one chunk, the embedding becomes a **blurry average** of all the topics in the document. It's like trying to describe a 300-page book in a single sentence — you lose all the specific details. Smaller chunks produce **focused embeddings** that can be precisely matched to specific queries.

</details>

---

**Question 2:** What problem does overlapping chunking solve that fixed-size chunking doesn't?

<details>
<summary>👉 Click to reveal answer</summary>

Overlapping chunking solves the **boundary information loss** problem. With fixed-size chunking, a sentence split between two chunks means neither chunk has the complete thought. With overlap, the shared region ensures that **boundary information appears in both chunks**, so it can always be found during retrieval.

</details>

---

**Question 3:** When would you choose semantic chunking over sentence-based chunking?

<details>
<summary>👉 Click to reveal answer</summary>

Choose **semantic chunking** when your text covers **multiple topics** and you want each chunk to be about **one coherent topic**. Sentence-based chunking groups sentences by size, so a chunk might contain sentences about two different topics if they happen to be adjacent. Semantic chunking detects **topic boundaries** and splits there, keeping each chunk topically focused.

</details>

---

**Question 4:** Why is 200-500 tokens considered the "sweet spot" for chunk size?

<details>
<summary>👉 Click to reveal answer</summary>

This range balances two competing needs: **enough context** to be meaningful (a 50-token chunk might just say "water evaporates" without explaining why) and **enough focus** to produce a precise embedding (a 2000-token chunk covers too many ideas). At 200-500 tokens, chunks typically contain **one complete idea or paragraph**, making them ideal for both embedding quality and retrieval precision.

</details>

---

**Question 5:** In recursive chunking, why do we try paragraph separators before sentence separators?

<details>
<summary>👉 Click to reveal answer</summary>

Paragraph breaks (`\n\n`) are the **most meaningful** separators because they typically indicate a **topic or section change**. By splitting on paragraphs first, we preserve the document's intended structure. We only fall back to sentence or word splitting when a single paragraph is still too large for our target chunk size. This preserves the **hierarchy of meaning** in the document.

</details>

---

## What's Next?

You've collected all five chunking strategies. Now you need somewhere to store these chunks and a way to search them at scale.

The next notebook covers vector databases: the specialized storage layer that makes retrieval fast across millions of chunks.

[Notebook 03: Vector Databases — Storing and Searching Your Chunks](03_vector_databases.ipynb)

You'll learn:
- What makes vector databases different from regular databases
- How approximate nearest neighbor search works at scale
- Popular options: FAISS, Chroma, Pinecone
- How to build a working vector store from scratch

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)